In [ ]:
import time
import json
import os
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from webdriver_manager.chrome import ChromeDriverManager

# Set up Selenium WebDriver with WebDriver Manager
chrome_options = Options()
chrome_options.add_argument("--headless")  # Run in headless mode
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

# Automatically download and use the correct ChromeDriver
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=chrome_options)

# File where data will be stored incrementally
JSON_FILE = "cisa_advisories.json"

# Load existing data if available
if os.path.exists(JSON_FILE):
    with open(JSON_FILE, "r", encoding="utf-8") as f:
        advisories = json.load(f)
else:
    advisories = []

# Set of already scraped URLs to avoid duplicates
scraped_urls = {entry["url"] for entry in advisories}

# # Set up Selenium WebDriver
# chrome_options = Options()
# chrome_options.add_argument("--headless")  # Run in headless mode
# chrome_options.add_argument("--no-sandbox")
# chrome_options.add_argument("--disable-dev-shm-usage")

# service = Service("chromedriver")  # Adjust path if necessary
# driver = webdriver.Chrome(service=service, options=chrome_options)
wait = WebDriverWait(driver, 10)

# Base URL
base_url = "https://www.cisa.gov/news-events/cybersecurity-advisories"
driver.get(base_url)

while True:
    try:
        # Wait for advisories to load
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.view-content")))

        # Find all advisory links
        advisory_links = driver.find_elements(By.CSS_SELECTOR, "div.views-row a[href*='/news-events/']")
        links = [link.get_attribute("href") for link in advisory_links if link.get_attribute("href") not in scraped_urls]

        for link in links:
            driver.get(link)
            time.sleep(2)  # Allow time for page load

            try:
                title = driver.find_element(By.CSS_SELECTOR, "h1").text
                date = driver.find_element(By.CSS_SELECTOR, "span.date-display-single").text if driver.find_elements(By.CSS_SELECTOR, "span.date-display-single") else "N/A"

                # Extract all paragraphs of advisory content
                content_sections = driver.find_elements(By.CSS_SELECTOR, "div.field--name-body p")
                content = "\n".join([section.text for section in content_sections if section.text.strip()])

                advisory_data = {
                    "title": title,
                    "date": date,
                    "content": content,
                    "url": link
                }

                advisories.append(advisory_data)
                scraped_urls.add(link)

                print(advisories)
                # Save after each advisory to prevent loss of data
                with open(JSON_FILE, "w", encoding="utf-8") as f:
                    json.dump(advisories, f, indent=4)

                print(f"Saved: {title}")

            except Exception as e:
                print(f"Error extracting advisory: {e}")

            # Go back to the list page
            driver.back()
            time.sleep(2)

        # Check if the "Next" button exists
        next_button = driver.find_elements(By.CSS_SELECTOR, "li.pager-next a")
        if next_button:
            next_button[0].click()
            time.sleep(3)  # Allow time for page transition
        else:
            print("No more pages to scrape.")
            break

    except Exception as e:
        print(f"Error processing page: {e}")
        break

# Save data as CSV (for easy viewing)
df = pd.DataFrame(advisories)
df.to_csv("cisa_advisories.csv", index=False)

print(f"Scraped {len(advisories)} advisories. Data saved to 'cisa_advisories.json' and 'cisa_advisories.csv'.")

# Close WebDriver
driver.quit()


WebDriverException: Message: unknown error: cannot find Chrome binary
Stacktrace:
#0 0x561ac0b9b4e3 <unknown>
#1 0x561ac08cac76 <unknown>
#2 0x561ac08f1757 <unknown>
#3 0x561ac08f0029 <unknown>
#4 0x561ac092eccc <unknown>
#5 0x561ac092e47f <unknown>
#6 0x561ac0925de3 <unknown>
#7 0x561ac08fb2dd <unknown>
#8 0x561ac08fc34e <unknown>
#9 0x561ac0b5b3e4 <unknown>
#10 0x561ac0b5f3d7 <unknown>
#11 0x561ac0b69b20 <unknown>
#12 0x561ac0b60023 <unknown>
#13 0x561ac0b2e1aa <unknown>
#14 0x561ac0b846b8 <unknown>
#15 0x561ac0b84847 <unknown>
#16 0x561ac0b94243 <unknown>
#17 0x7f6b34ef5ac3 <unknown>


In [19]:
import os
from natsort import natsorted
import pickle as pkl

In [13]:
dump=os.listdir('articles')
print(len(dump))

3373


In [14]:
dump=natsorted(dump)

In [15]:
dump[0]

'ics-alert-10-194-01'

In [16]:
text=[]
for items in dump:
    with open('articles/{}'.format(items),'r') as f:
        file=f.read().strip()

    text.append(file)

In [27]:
text[2000]

'1. EXECUTIVE SUMMARY\n\nCVSS v3 5.5\nATTENTION: Low attack complexity\nVendor: Rockwell Automation\nEquipment: ISaGRAF\nVulnerability: Improper Restriction of XML External Entity Reference\n\n2.UPDATE INFORMATION\nThis updated advisory is a follow-up to the advisory update titled ICSA-22-088-01 Rockwell Automation ISaGRAF that was published March 29, 2022, to the ICS webpage at www.cisa.gov/ics.\n3. RISK EVALUATION\nSuccessful exploitation of this vulnerability could allow an attacker to pass local file data to a remote web server, leading to loss of confidentiality.\n4. TECHNICAL DETAILS\n4.1 AFFECTED PRODUCTS\nThe following Rockwell Automation software products are affected:\n\nConnected Component Workbench: v12.00 and prior\n\n--------- Begin Update A Part 1 of 2 ---------\n\nISaGRAF Workbench: All versions prior to v6.6.10\n\n--------- End Update A Part 1 of 2 ---------\n\nISaGRAF Workbench: v6.6.9 and prior\nSafety Instrumented Systems Workstation: v1.1 and prior\n\n4.2 VULNERABI

In [25]:
with open('outputs/9.pkl','rb') as f:
    file=pkl.load(f)

print(file)

['[[Software: "Ecava IntegraXor", Vulnerability: "Directory Traversal", CVE: "", CWE: ""]]']


In [29]:
dump=os.listdir('outputs')
print(len(dump))

461
